# Principia Online Retrieval and Embedding Rerank Tutorial

## Goal

This notebook runs the full principia_retrieval pipeline against live public literature sources. It uses an LLM to plan academic search queries, retrieves work metadata from arXiv, OpenAlex, Crossref, and Semantic Scholar, normalizes and deduplicates candidates, ranks with BM25, then reranks the BM25 head with live embeddings.

It makes external requests and requires a SiliconFlow API key. The key is used for both query planning and embedding reranking; public metadata sources themselves do not require one.

## Setup

Install the framework with pip install principia-ai ipykernel, or install the repository version from Principia-v1.3 with python -m pip install -e ".[dev]".

Configure the provider key in your shell, never in this notebook:

    export SILICONFLOW_API_KEY="sk-..."

Running the notebook makes live provider requests and may incur cost.

In [ ]:
from __future__ import annotations

import os

import principia as pc
from principia_retrieval import RetrievalConfig, WorkRetriever
from principia_retrieval.embeddings import SiliconFlowEmbeddingClient

API_KEY = os.environ.get("SILICONFLOW_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError(
        "Set SILICONFLOW_API_KEY before running this online retrieval tutorial."
    )

GOAL = (
    "Please design an MAS framework where LLMs are interacting machine dialects "
    "like social interaction to improve the reasoning accuracy and reducing the tokens completion and cost."
)
SOURCE_NAMES = ["arxiv", "openalex", "crossref", "semantic_scholar"]
TARGET_COUNT = 8
MAX_RAW_CANDIDATES = 48

PLANNER_MODEL = "siliconflow:Qwen/Qwen3.5-397B-A17B"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"
EMBEDDING_DIMENSIONS = 1024

llm = pc.LLMClient(
    pc.siliconflow_config(API_KEY, model=PLANNER_MODEL, timeout=60, max_retries=2)
)
embedding_client = SiliconFlowEmbeddingClient(
    api_key=API_KEY,
    model=EMBEDDING_MODEL,
    dimensions=EMBEDDING_DIMENSIONS,
    timeout=45,
    max_retries=2,
)

assert llm.available()
assert embedding_client.available()
print(f"Principia {pc.__version__}; sources={', '.join(SOURCE_NAMES)}")

## Steps

### 1. Retrieve and rank with BM25

WorkRetriever uses the configured LLM to produce a bounded academic query plan. It fans those queries out to the selected public metadata sources, normalizes and deduplicates scholarly records, and ranks candidates deterministically with BM25. The callback exposes the query plan and stage diagnostics.

In [ ]:
bm25_events: list[tuple[str, dict]] = []

bm25_retriever = WorkRetriever(
    config=RetrievalConfig(
        use_llm_planner=True,
        rerank_mode="bm25",
        source_names=SOURCE_NAMES,
        max_queries=6,
        max_raw_candidates=MAX_RAW_CANDIDATES,
        min_relevance=0.04,
    )
)
bm25_result = bm25_retriever.search(
    GOAL,
    target_count=TARGET_COUNT,
    llm=llm,
    timeout=20,
    callback=lambda stage, payload: bm25_events.append((stage, payload)),
)

print("Planned queries:")
for query in bm25_result.query_plan.search_queries:
    print("-", query)
print("Entities:", bm25_result.query_plan.entities)
print("Candidates after normalization/deduplication:", len(bm25_result.candidates))
print("Selected BM25 works:", len(bm25_result.selected_works))

In [ ]:
def trace_rows(result):
    return [
        {
            "rank": index + 1,
            "work_id": row["work_id"],
            "score": row["score"],
            "relation": row["relation_label"],
            "title": row["title"],
            "rationale": row["rationale"],
        }
        for index, row in enumerate(result.ranking_trace)
    ]

trace_rows(bm25_result)

### 2. Rerank the BM25 head with embeddings

Embedding reranking begins with the same BM25 prefilter. Only the bounded head is sent to the injected SiliconFlowEmbeddingClient. This keeps embedding cost and latency bounded while allowing semantic similarity to change the final order.

If the embedding request fails, the retriever retains the BM25-ranked candidates and records an embedding_rerank_error signal instead of discarding the result.

In [ ]:
embedding_events: list[tuple[str, dict]] = []

embedding_retriever = WorkRetriever(
    config=RetrievalConfig(
        use_llm_planner=True,
        rerank_mode="embedding_rerank",
        source_names=SOURCE_NAMES,
        max_queries=6,
        max_raw_candidates=MAX_RAW_CANDIDATES,
        min_relevance=0.04,
        embedding_model=EMBEDDING_MODEL,
        embedding_dimensions=EMBEDDING_DIMENSIONS,
        embedding_batch_size=24,
        embedding_timeout=45,
        embedding_max_retries=2,
        embedding_rerank_candidate_limit=32,
    )
)
embedding_result = embedding_retriever.search(
    GOAL,
    target_count=TARGET_COUNT,
    llm=llm,
    timeout=20,
    embedding_client=embedding_client,
    callback=lambda stage, payload: embedding_events.append((stage, payload)),
)

trace_rows(embedding_result)

### 3. Inspect diagnostics and compare the two rankings

Each ranking trace exposes the work identifier, score, relation label, and rationale. The diagnostics event measures the raw source responses, normalized/deduplicated candidates, BM25 prefilter, embedding input, and final selected counts.

In [ ]:
def event_payload(events, stage):
    return next(payload for event, payload in events if event == stage)

bm25_diagnostics = event_payload(bm25_events, "retrieval_diagnostics")
embedding_diagnostics = event_payload(embedding_events, "retrieval_diagnostics")

print("BM25 selected IDs:", [work["work_id"] for work in bm25_result.selected_works])
print("Embedding-reranked IDs:", [work["work_id"] for work in embedding_result.selected_works])
print("BM25 diagnostics:", bm25_diagnostics)
print("Embedding diagnostics:", embedding_diagnostics)

In [ ]:
if not bm25_result.selected_works:
    raise RuntimeError(
        "No BM25 results were returned. Check public-source connectivity, source availability, and the goal."
    )
if not embedding_result.selected_works:
    raise RuntimeError(
        "No embedding-reranked results were returned. Check the embedding provider and source connectivity."
    )

assert bm25_diagnostics["rerank_mode"] == "bm25"
assert embedding_diagnostics["rerank_mode"] == "embedding_rerank"
assert embedding_diagnostics["embedding_input_count"] <= 32
assert embedding_diagnostics["selected_count"] == len(embedding_result.selected_works)
print("Online retrieval and embedding-rerank checks passed.")

## Checks

A successful run produces an LLM-generated query plan, a raw candidate count greater than or equal to the deduplicated count, zero embedding input for BM25, a bounded nonzero embedding input for reranking, and a trace for every selected work.

If embeddings are unavailable, inspect community_signals on selected works for embedding_rerank_error. The remaining ordering is the usable BM25 fallback.

## Next Steps

- Run deterministic retrieval without provider calls by setting use_llm_planner to False and rerank_mode to bm25.
- Use pc.Workspace(...).research.search(...) when you need the V1.3 framework wrapper; it delegates to the same package.
- Compare ordering within a rerank mode, not the absolute score values between BM25 and embedding modes.
- Record retrieval diagnostics in production telemetry before changing source, candidate, or rerank budgets.